# 01 - Define and Generate CNV Templates

This notebook creates multiple CNV templates for simulation while keeping the same 4-subclone structure:
- `N`: normal (no CNVs)
- `A`: trunk CNVs (+ optional A-specific CNVs)
- `B`: A + B-specific CNVs
- `C`: A + C-specific CNVs

Across templates, we vary:
1. Number of CNVs defining the subclones
2. CNV size range (`min_size_bp`, `max_size_bp`)

Outputs per template:
- CNV interval CSV: `data/templates/CNVs_<template_id>.csv`
- Subclone definition JSON: `manifests/subclone_<template_id>.json`
- Template manifest: `manifests/templates.csv`

In [1]:
from pathlib import Path
import json
import random
import pandas as pd
import scanpy as sc
import sys
sys.path.append("/home/augusta/storage3/augusta/insituCNV/InSituCNV")
import insitucnv as icv

In [2]:
# Root for the new technical-constraints workflow
ROOT = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains')
TEMPLATE_DIR = ROOT / 'data' / 'templates'
MANIFEST_DIR = ROOT / 'manifests'
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

GENE_INFO = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Ensmbl_BioMart_gene_info.txt')
BASE_ADATA = Path('/home/augusta/storage3/augusta/insituCNV/data/simulated_CNV_data/lung_organoids_cnvclust.h5ad')

assert GENE_INFO.exists(), f'Missing gene-info file: {GENE_INFO}'
assert BASE_ADATA.exists(), f'Missing base AnnData file: {BASE_ADATA}'

In [3]:
# Optional sanity check against available genes in base adata
adata = sc.read_h5ad(BASE_ADATA)
adata

AnnData object with n_obs × n_vars = 1268 × 25691
    obs: 'organism_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'model_id', 'sample_id', 'Phase', 'level_1', 'level_2', 'level_3', 'CountUMIs', 'CountGenes', 'X.Mitochondrial', 'NoveltyScore', 'nCount_SCT', 'nFeature_SCT', 'orig.ident', 'is_primary_data', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'cnv_leiden'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype',

In [4]:
# Pools used to draw template-specific trunk CNVs
TRUNK_GAIN_POOL = [
    'ERBB2', 'EGFR', 'MYC', 'CCND1', 'MDM2', 'CDK4', 'MET', 'FGFR1'
]
TRUNK_LOSS_POOL = [
    'DIS3', 'MECOM', 'TP53', 'RB1', 'PTEN', 'KEAP1', 'NF1', 'CDKN2A'
]

# Candidate pools for additional branch-defining events
GAIN_POOL = [
    'CHD7', 'HCK', 'MYD88', 'TBX3', 'EGFR', 'MYC', 'CCND1',
    'MDM2', 'CDK4', 'MET', 'FGFR1', 'PIK3CA', 'NFE2L2', 'ALK'
]

LOSS_POOL = [
    'KEAP1', 'TP53', 'RB1', 'PTEN', 'NF1', 'CDKN2A',
    'SMAD4', 'ARID1A', 'ATM', 'BAP1', 'STK11', 'APC'
]

print('Trunk gain candidates:', len(TRUNK_GAIN_POOL), '| trunk loss candidates:', len(TRUNK_LOSS_POOL))
print('Branch gain candidates:', len(GAIN_POOL), '| branch loss candidates:', len(LOSS_POOL))


Trunk gain candidates: 8 | trunk loss candidates: 8
Branch gain candidates: 14 | branch loss candidates: 12


In [5]:
import numpy as np

def build_trunk_for_template(template_id, n_trunk=3, seed=1234):
    """Create a reproducible trunk per template with both gain and loss events."""
    if n_trunk < 2:
        raise ValueError('n_trunk must be >= 2 so trunk has both gain and loss.')

    tid_num = int(str(template_id).replace('T', ''))
    rng = np.random.default_rng(seed + tid_num)

    n_gain = int(rng.integers(1, n_trunk))
    n_loss = n_trunk - n_gain

    if n_gain > len(TRUNK_GAIN_POOL) or n_loss > len(TRUNK_LOSS_POOL):
        raise ValueError('Requested n_trunk exceeds available trunk pools.')

    trunk_gain = rng.choice(TRUNK_GAIN_POOL, size=n_gain, replace=False).tolist()
    trunk_loss = rng.choice(TRUNK_LOSS_POOL, size=n_loss, replace=False).tolist()

    trunk = {g: 'gain' for g in trunk_gain}
    trunk.update({g: 'loss' for g in trunk_loss})
    return trunk

# Example
for tid in ['T01', 'T02', 'T03', 'T04']:
    print(tid, build_trunk_for_template(tid, n_trunk=3, seed=1234))


T01 {'EGFR': 'gain', 'CDKN2A': 'loss', 'MECOM': 'loss'}
T02 {'MDM2': 'gain', 'EGFR': 'gain', 'KEAP1': 'loss'}
T03 {'CCND1': 'gain', 'FGFR1': 'gain', 'DIS3': 'loss'}
T04 {'ERBB2': 'gain', 'MET': 'gain', 'MECOM': 'loss'}


## Template specification
`n_A_unique`, `n_B_unique`, `n_C_unique` control how many extra CNVs define each subclone branch.
`min_size_bp`, `max_size_bp` control CNV interval size range.

In [6]:
import numpy as np
import pandas as pd

# Keep your CNV-size / event settings, but remove fixed probabilities from tuples
template_specs_base = [
    # template_id, min_size_bp, max_size_bp, n_A_unique, n_B_unique, n_C_unique, seed
    ('T01', 10_000_000, 20_000_000, 0, 3, 3, 1001),
    ('T02',  5_000_000, 10_000_000, 1, 2, 2, 1002),
    ('T03',  2_000_000,  8_000_000, 1, 4, 4, 1003),
    ('T04',  1_000_000,  5_000_000, 2, 3, 3, 1004),
    ('T05',  8_000_000, 15_000_000, 2, 5, 5, 1005),
    ('T06',  3_000_000, 12_000_000, 2, 4, 6, 1006),
    ('T07',    800_000,  3_000_000, 3, 4, 4, 1007),
    ('T08', 15_000_000, 35_000_000, 1, 3, 5, 1008),
    ('T09',  4_000_000, 16_000_000, 3, 5, 3, 1009),
    ('T10',  6_000_000, 22_000_000, 2, 6, 6, 1010),
]

def sample_probs_with_bounds(seed, min_p=0.01, max_p=0.90):
    rng = np.random.default_rng(seed)
    while True:
        p = rng.dirichlet(np.ones(4))
        if (p >= min_p).all() and (p <= max_p).all():
            return p  # [p_N, p_A, p_B, p_C]

rows = []
for t in template_specs_base:
    template_id, min_bp, max_bp, nA, nB, nC, seed = t
    pN, pA, pB, pC = sample_probs_with_bounds(seed, min_p=0.01, max_p=0.90)
    rows.append((template_id, min_bp, max_bp, nA, nB, nC, pN, pA, pB, pC, seed))

spec_df = pd.DataFrame(rows, columns=[
    'template_id', 'min_size_bp', 'max_size_bp',
    'n_A_unique', 'n_B_unique', 'n_C_unique',
    'p_N', 'p_A', 'p_B', 'p_C', 'seed'
])

# Validate probability rows
prob_sums = spec_df[['p_N', 'p_A', 'p_B', 'p_C']].sum(axis=1)
if not np.allclose(prob_sums.values, 1.0):
    bad = spec_df.loc[~np.isclose(prob_sums.values, 1.0), ['template_id', 'p_N', 'p_A', 'p_B', 'p_C']]
    raise ValueError(f'Subclone probabilities must sum to 1.0. Bad rows:\n{bad}')

spec_df


,template_id,min_size_bp,max_size_bp,n_A_unique,n_B_unique,n_C_unique,p_N,p_A,p_B,p_C,seed
0,T01,10000000,20000000,0,3,3,0.358766,0.012329,0.095003,0.533902,1001
1,T02,5000000,10000000,1,2,2,0.132187,0.307734,0.378655,0.181424,1002
2,T03,2000000,8000000,1,4,4,0.079636,0.323159,0.355869,0.241336,1003
3,T04,1000000,5000000,2,3,3,0.061255,0.717915,0.055858,0.164972,1004
4,T05,8000000,15000000,2,5,5,0.018557,0.402108,0.080729,0.498606,1005
5,T06,3000000,12000000,2,4,6,0.195534,0.683437,0.068393,0.052635,1006
6,T07,800000,3000000,3,4,4,0.046973,0.563726,0.293425,0.095875,1007
7,T08,15000000,35000000,1,3,5,0.141423,0.254874,0.169610,0.434093,1008
8,T09,4000000,16000000,3,5,3,0.094236,0.226527,0.142164,0.537073,1009
9,T10,6000000,22000000,2,6,6,0.022548,0.279946,0.324116,0.373390,1010


In [7]:
def sample_unique_events(n_events, gain_pool, loss_pool, rng):
    """Return dict {gene: type} with roughly balanced gains/losses."""
    if n_events == 0:
        return {}

    n_gain = n_events // 2
    n_loss = n_events - n_gain

    if n_gain > len(gain_pool) or n_loss > len(loss_pool):
        raise ValueError('Not enough genes in pools for requested event count.')

    gain_genes = rng.sample(gain_pool, n_gain)
    loss_genes = rng.sample(loss_pool, n_loss)

    events = {g: 'gain' for g in gain_genes}
    events.update({g: 'loss' for g in loss_genes})
    return events


def build_template_subclones(row):
    """Build CNV_dict and 4-subclone definition for one template spec row."""
    rng = random.Random(int(row.seed))

    # Template-specific trunk (different across templates, reproducible)
    trunk = build_trunk_for_template(row.template_id, n_trunk=3, seed=int(row.seed))

    # Avoid overlap by removing already-used genes from next samples
    used = set(trunk.keys())

    gain_available = [g for g in GAIN_POOL if g not in used]
    loss_available = [g for g in LOSS_POOL if g not in used]
    A_unique = sample_unique_events(int(row.n_A_unique), gain_available, loss_available, rng)
    used.update(A_unique.keys())

    gain_available = [g for g in GAIN_POOL if g not in used]
    loss_available = [g for g in LOSS_POOL if g not in used]
    B_unique = sample_unique_events(int(row.n_B_unique), gain_available, loss_available, rng)
    used.update(B_unique.keys())

    gain_available = [g for g in GAIN_POOL if g not in used]
    loss_available = [g for g in LOSS_POOL if g not in used]
    C_unique = sample_unique_events(int(row.n_C_unique), gain_available, loss_available, rng)

    # Hierarchical subclone definitions
    A_events = dict(trunk)
    A_events.update(A_unique)

    B_events = dict(A_events)
    B_events.update(B_unique)

    C_events = dict(A_events)
    C_events.update(C_unique)

    subclone_dict = {
        'N': [],
        'A': list(A_events.keys()),
        'B': list(B_events.keys()),
        'C': list(C_events.keys()),
    }

    # Global CNV dict used by generate_cnvs
    cnv_dict = {}
    cnv_dict.update(A_events)
    cnv_dict.update(B_unique)
    cnv_dict.update(C_unique)

    return cnv_dict, subclone_dict, trunk


In [8]:
# Check gene availability in the source adata before generating templates
adata_genes = set(adata.var['gene_symbols'].astype(str)) if 'gene_symbols' in adata.var.columns else set(adata.var_names.astype(str))

all_selected_genes = set(TRUNK_GAIN_POOL) | set(TRUNK_LOSS_POOL) | set(GAIN_POOL) | set(LOSS_POOL)
missing_in_adata = sorted(g for g in all_selected_genes if g not in adata_genes)
print(f'Selected gene pool size: {len(all_selected_genes)}')
print(f'Missing in base adata: {len(missing_in_adata)}')
if missing_in_adata:
    print('Examples missing:', missing_in_adata[:20])


Selected gene pool size: 29
Missing in base adata: 0


In [9]:
manifest_rows = []

for row in spec_df.itertuples(index=False):
    template_id = row.template_id

    cnv_dict, subclone_dict, trunk = build_template_subclones(row)

    cnv_csv = TEMPLATE_DIR / f'CNVs_{template_id}.csv'
    subclone_json = MANIFEST_DIR / f'subclone_{template_id}.json'

    # Generate CNV intervals with template-specific size range
    cnv_df = icv.pp.generate_cnvs(
        cnv_dict,
        min_size=int(row.min_size_bp),
        max_size=int(row.max_size_bp),
        gene_info=str(GENE_INFO),
        save_csv=str(cnv_csv),
    )

    # Persist subclone definitions
    with open(subclone_json, 'w') as f:
        json.dump(subclone_dict, f, indent=2)

    manifest_rows.append({
        'template_id': template_id,
        'template_name': f'template_{template_id.lower()}',
        'source_cnv_csv': str(cnv_csv),
        'source_subclone_json': str(subclone_json),
        'seed': int(row.seed),
        'min_size_bp': int(row.min_size_bp),
        'max_size_bp': int(row.max_size_bp),
        'n_A_unique': int(row.n_A_unique),
        'n_B_unique': int(row.n_B_unique),
        'n_C_unique': int(row.n_C_unique),
        'n_total_cnv_genes': int(len(cnv_dict)),
        'n_trunk': int(len(trunk)),
        'trunk_genes': ';'.join(sorted(trunk.keys())),
        'p_N': float(row.p_N),
        'p_A': float(row.p_A),
        'p_B': float(row.p_B),
        'p_C': float(row.p_C),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df = manifest_df.sort_values('template_id').reset_index(drop=True)
manifest_df

,template_id,template_name,source_cnv_csv,source_subclone_json,seed,min_size_bp,max_size_bp,n_A_unique,n_B_unique,n_C_unique,n_total_cnv_genes,n_trunk,trunk_genes,p_N,p_A,p_B,p_C
0,T01,template_t01,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1001,10000000,20000000,0,3,3,9,3,CDKN2A;MET;MYC,0.358766,0.012329,0.095003,0.533902
1,T02,template_t02,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1002,5000000,10000000,1,2,2,8,3,ERBB2;FGFR1;MECOM,0.132187,0.307734,0.378655,0.181424
2,T03,template_t03,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1003,2000000,8000000,1,4,4,12,3,EGFR;KEAP1;MYC,0.079636,0.323159,0.355869,0.241336
3,T04,template_t04,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1004,1000000,5000000,2,3,3,11,3,EGFR;NF1;RB1,0.061255,0.717915,0.055858,0.164972
4,T05,template_t05,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1005,8000000,15000000,2,5,5,15,3,CDK4;DIS3;ERBB2,0.018557,0.402108,0.080729,0.498606
5,T06,template_t06,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1006,3000000,12000000,2,4,6,15,3,EGFR;MECOM;MYC,0.195534,0.683437,0.068393,0.052635
6,T07,template_t07,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1007,800000,3000000,3,4,4,14,3,CCND1;CDKN2A;FGFR1,0.046973,0.563726,0.293425,0.095875
7,T08,template_t08,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1008,15000000,35000000,1,3,5,12,3,DIS3;EGFR;FGFR1,0.141423,0.254874,0.169610,0.434093
8,T09,template_t09,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1009,4000000,16000000,3,5,3,14,3,CDKN2A;EGFR;MDM2,0.094236,0.226527,0.142164,0.537073
9,T10,template_t10,/home/augusta/storage3/augusta/insituCNV/InSit...,/home/augusta/storage3/augusta/insituCNV/InSit...,1010,6000000,22000000,2,6,6,17,3,MDM2;MET;TP53,0.022548,0.279946,0.324116,0.373390


In [10]:
templates_manifest_path = MANIFEST_DIR / 'templates.csv'
manifest_df.to_csv(templates_manifest_path, index=False)
print(f'Saved template manifest: {templates_manifest_path}')

Saved template manifest: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/manifests/templates.csv


In [11]:
# Optional quick look at one template
example_template = 'T01'
display(pd.read_csv(TEMPLATE_DIR / f'CNVs_{example_template}.csv').head())
with open(MANIFEST_DIR / f'subclone_{example_template}.json') as f:
    print(json.dumps(json.load(f), indent=2))

,Gene name,Chromosome,Size (bp),Type,Start (bp),End (bp)
0,MET,7,10634046,gain,111418263,122052309
1,MYC,8,17658722,gain,118909831,136568553
2,CDKN2A,9,13931824,loss,15015614,28947438
3,CHD7,8,19775454,gain,50885657,70661111
4,PTEN,10,12388600,loss,81722984,94111584


{
  "N": [],
  "A": [
    "MET",
    "MYC",
    "CDKN2A"
  ],
  "B": [
    "MET",
    "MYC",
    "CDKN2A",
    "CHD7",
    "PTEN",
    "TP53"
  ],
  "C": [
    "MET",
    "MYC",
    "CDKN2A",
    "PIK3CA",
    "BAP1",
    "NF1"
  ]
}
